# Convención del pLDDT de Boltz-2 y Chai-1, con archivos reales

**Qué mide.** MolDesign sólo lee el pLDDT de una estructura predicha si la convención de su fuente está verificada
con un archivo real (`backend/services/targets/prediccion.py`). Hoy sólo lo está la de AlphaFold DB: una estructura
de Boltz o de Chai-1 sale con el pLDDT «no disponible». Este cuaderno genera esos archivos y compara cada campo
candidato del archivo (factor B por átomo, media por residuo, `_ma_qa_metric_local`) con el pLDDT que devuelve el
propio modelo: el `plddt_*.npz` de Boltz y el tensor `plddt` que devuelve `run_inference` en Chai-1.

**Criterio, fijado antes de ejecutar.** La convención queda verificada para un grupo (herramienta, formato) si se
cumplen las dos condiciones:

- el grupo tiene **al menos dos estructuras** terminadas;
- en **todas** ellas, un mismo campo del archivo reproduce el pLDDT del modelo residuo a residuo (o átomo a átomo).
  Debe hacerlo con **una sola escala** (×1, ×100 o ×0,01) y con una diferencia que no supere el redondeo con que el
  archivo escribe el número.

Si una sola estructura no cumple, el grupo queda sin verificar. Lo que no termina (por memoria o por bf16) se
registra y no cuenta ni a favor ni en contra. La correlación con AlphaFold DB es un control de sentido y no forma
parte del criterio.

**Casos.**

| Caso | Boltz | Chai-1 |
|---|---|---|
| CDK2 sin ligando (P24941) | mmCIF y PDB | mmCIF, 5 muestras |
| CDK2 con un ligando del benchmark de OpenFF | mmCIF y PDB | mmCIF, 5 muestras |
| NET humano (P23975, la misma proteína del fixture de AlphaFold DB del producto) | mmCIF | mmCIF, 5 muestras |

**Qué sale de Colab.** No sale nada del repositorio ni de MolDesign. Sólo se conecta con estos servicios públicos:

- UniProt (`rest.uniprot.org`), de donde se descargan las secuencias;
- el servidor MSA de ColabFold (`api.colabfold.com`), que recibe esas secuencias si `USAR_SERVIDOR_MSA` está activado;
- AlphaFold DB (EBI), de donde se descarga el modelo de P23975;
- PyPI y los sitios de donde cada modelo descarga sus pesos.

**Entorno.** Elige L4 en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Chai-1 necesita bf16, así
que en una T4 probablemente no corra. Una A100 también sirve, pero gasta más unidades por hora.

**Coste.** Es una estimación, no una medida: la instalación y las descargas llevan unos 20–30 min y las predicciones
unos 20–45 min. En total, alrededor de una hora de L4.

**Cómo ejecutarlo.**

1. Elige *Entorno de ejecución → Ejecutar todo*.
2. El cuaderno pide permiso para montar tu Drive, donde copia los resultados después de cada paso. Si Colab se
   desconecta, o si quieres repetir lo que falló, vuelve a ejecutarlo todo. Retoma los resultados de Drive, instala y
   corre sólo las herramientas con corridas pendientes, y salta las que ya terminaron. Lo que falló se aparta con su
   fecha, aquí y en Drive; no se borra nada.
3. Al final descarga un zip.
4. **Desconecta el entorno** en cuanto termine, porque una GPU conectada gasta unidades aunque no esté haciendo nada.

In [ ]:
# ── Parámetros. Están fijados antes de ejecutar; si cambias alguno, queda registrado en procedencia.json ──
SEMILLA = 42
USAR_SERVIDOR_MSA = True    # envía SÓLO las secuencias públicas de UniProt a api.colabfold.com
# Fijadas a las que resolvió la primera ejecución (2026-11-14): una repetición no puede cambiar de versión sin decirlo.
# "" = la última de PyPI (se registra la exacta).
VERSION_BOLTZ = "2.2.1"
VERSION_CHAI = "0.6.1"
MUESTRAS_CHAI = 5
GUARDAR_EN_DRIVE = True     # copia los resultados a tu Drive tras cada paso (sobreviven a una desconexión)

LIGANDO_CDK2 = {
    "smiles": "OCc1cccc(Nc2nc(OCC3CCCCC3)c3nc[nH]c3n2)c1",
    "origen": "ligando de cdk2 del benchmark de OpenFF, tal como lo envió FEP-09-R1 "
              "(scripts/artifacts_science/FEP-09-R1/_work/pilot/evidence/cdk2__baseline__0/submit_request.json)",
}
CASOS = [
    {"id": "cdk2_apo", "uniprot": "P24941", "ligando": None},
    {"id": "cdk2_lig", "uniprot": "P24941", "ligando": LIGANDO_CDK2},
    {"id": "net_apo", "uniprot": "P23975", "ligando": None},  # NET humano, la proteína de AF-P23975 (fixture del producto)
]
FORMATOS_BOLTZ = {"cdk2_apo": ["mmcif", "pdb"], "cdk2_lig": ["mmcif", "pdb"], "net_apo": ["mmcif"]}
REFERENCIA_AFDB = {
    "url": "https://alphafold.ebi.ac.uk/files/AF-P23975-F1-model_v6.cif",
    # el hash del original que recortaron los fixtures (backend/tests/data/receptores_propios/alphafold_procedencia.json)
    "sha256_en_el_producto": "e2f921b2b10d499aa1f8d5aebefb3f1c53a6cee0b95c755dbb3a7e5a048fd899",
}

In [ ]:
# ── Entorno: GPU, Drive y registro de procedencia ──
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

RAIZ = Path("/content/plddt_boltz_chai")
RES = RAIZ / "resultados"
CACHE = Path("/content/cache")
VENVS = Path("/content/venvs")
DRIVE = Path("/content/drive/MyDrive/moldesign_plddt_boltz_chai/resultados")
for d in (RAIZ, RES, CACHE, VENVS):
    d.mkdir(parents=True, exist_ok=True)


def ahora():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def sello():
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")


#: Los entornos virtuales no heredan lo que Colab pone para su propio Python: PYTHONPATH mezclaría sus paquetes, y
#: su MPLBACKEND (matplotlib_inline) no existe en ellos, así que cualquier gráfico de Chai-1 fallaría al importarse.
ENTORNO_LIMPIO = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "MPLBACKEND")}
ENTORNO_LIMPIO["MPLBACKEND"] = "Agg"


def sh(cmd, log=None, env=None, check=False):
    """Ejecuta y devuelve (resultado, segundos). Con ``log``, la salida va a ese archivo y no a la celda."""
    env = ENTORNO_LIMPIO if env is None else env
    t0 = time.time()
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, env=env)
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if check and r.returncode != 0:
        detalle = (r.stdout or "") + (r.stderr or "") if not log else f"mira {log}"
        raise RuntimeError(f"falló {' '.join(map(str, cmd[:4]))}: {detalle[-3000:]}")
    return r, round(time.time() - t0, 1)


PROC_RUTA = RES / "procedencia.json"


def cargar_procedencia():
    return json.loads(PROC_RUTA.read_text(encoding="utf-8")) if PROC_RUTA.exists() else {}


def guardar_procedencia(**cambios):
    p = cargar_procedencia()
    p.update(cambios)
    PROC_RUTA.write_text(json.dumps(p, ensure_ascii=False, indent=2), encoding="utf-8")
    return p


def drive_montado():
    return GUARDAR_EN_DRIVE and Path("/content/drive/MyDrive").exists()


def respaldar():
    if drive_montado():
        shutil.copytree(RES, DRIVE, dirs_exist_ok=True)


def apartar(ruta, motivo="fallido"):
    """Aparta un resultado anterior renombrándolo con la fecha, aquí y en su copia de Drive. Nunca se borra nada."""
    ruta = Path(ruta)
    nuevo = ruta.with_name(f"{ruta.stem}__{motivo}_{sello()}{ruta.suffix}")
    if drive_montado():
        copia = DRIVE / ruta.relative_to(RES)
        if copia.exists():
            copia.rename(DRIVE / nuevo.relative_to(RES))
    ruta.rename(nuevo)
    return nuevo


CLAVES = {"boltz": [f"{c['id']}__{f}" for c in CASOS for f in FORMATOS_BOLTZ[c["id"]]],
          "chai": [f"{c['id']}__mmcif" for c in CASOS]}


def pendientes():
    """Las corridas que faltan, por herramienta. Una terminada (código 0 y su carpeta) no se repite."""
    p = cargar_procedencia()
    return {h: [k for k in claves
                if not (p.get(f"corridas_{h}", {}).get(k, {}).get("codigo") == 0 and (RES / h / k).exists())]
            for h, claves in CLAVES.items()}


CHAI_LISTO = None  # lo decide la celda de instalación tras sondear parquet en el entorno de Chai-1

if GUARDAR_EN_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    if DRIVE.exists() and not any(RES.iterdir()):
        shutil.copytree(DRIVE, RES, dirs_exist_ok=True)  # retoma lo ya hecho en una ejecución anterior
        print("Retomado desde Drive:", sorted(p.name for p in RES.iterdir()))

r, _ = sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit("No hay GPU. Entorno de ejecución → Cambiar tipo de entorno de ejecución → L4, y vuelve a ejecutar.")
nombre_gpu, memoria_gpu, driver = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
import torch  # el torch del propio Colab, sólo para leer la capacidad de la GPU

CAPACIDAD = float("%d.%d" % torch.cuda.get_device_capability(0))
gpu = {"nombre": nombre_gpu, "memoria": memoria_gpu, "driver": driver, "compute_capability": CAPACIDAD}
if CAPACIDAD < 8.0:
    print("Aviso: esta GPU es anterior a Ampere. Chai-1 pide bf16 y probablemente falle; Boltz irá sin kernels.")
parametros = {"semilla": SEMILLA, "msa_servidor": USAR_SERVIDOR_MSA, "version_boltz": VERSION_BOLTZ or "última",
              "version_chai": VERSION_CHAI or "última", "muestras_chai": MUESTRAS_CHAI, "casos": CASOS,
              "formatos_boltz": FORMATOS_BOLTZ, "referencia_afdb": REFERENCIA_AFDB}
previa = cargar_procedencia()
ejecuciones = previa.get("ejecuciones", [])
if not ejecuciones and previa.get("inicio"):  # la primera ejecución no llevaba esta lista: se reconstruye su entrada
    ejecuciones.append({"inicio": previa["inicio"], "fin": previa.get("fin"), "gpu": previa.get("gpu"),
                        "parametros": previa.get("parametros")})
ejecuciones.append({"inicio": ahora(), "gpu": gpu, "parametros": parametros, "pendientes_al_empezar": pendientes()})
guardar_procedencia(gpu=gpu, inicio=previa.get("inicio") or ahora(), python_del_cuaderno=sys.version.split()[0],
                    plataforma=platform.platform(), parametros=parametros, ejecuciones=ejecuciones)
print(gpu)
print("Pendiente:", pendientes())

In [ ]:
%%writefile /content/plddt_boltz_chai/analisis_plddt.py
"""Dónde guarda Boltz o Chai-1 el pLDDT en su archivo, comparado con lo que devuelve el propio modelo.

Por qué existe: ``backend/services/targets/prediccion.py`` sólo lee el pLDDT de una fuente cuya convención se haya
verificado con un archivo real. Hoy sólo AlphaFold DB lo está; Boltz y Chai-1 salen «no disponible» porque nadie tenía
un archivo suyo. El cuaderno ``convencion_plddt_boltz_chai.ipynb`` genera esos archivos en Colab y este módulo los
compara, campo a campo, con la salida numérica del modelo (Boltz: ``plddt_*.npz``; Chai-1: el tensor ``plddt`` que
devuelve ``run_inference``).

Criterio, fijado antes de ejecutar nada. Una convención queda **verificada** para un grupo (herramienta, formato) si:

* el grupo tiene al menos ``MINIMO_DE_ESTRUCTURAS`` estructuras terminadas, y
* en **todas** ellas un mismo campo del archivo reproduce el pLDDT del modelo, residuo a residuo (o átomo a átomo),
  con **una única escala** (×1, ×100 o ×0,01) y una diferencia máxima no mayor que el redondeo con que el archivo
  escribe el número (medio dígito del último decimal escrito).

Una sola estructura que no cumpla deja el grupo sin verificar. Lo que no termina (memoria, bf16) se registra y no
cuenta ni a favor ni en contra. La correlación con AlphaFold DB es un control de sentido, no parte del criterio.

Sólo usa la biblioteca estándar, numpy y gemmi: corre igual en Colab y en el ``python-embed`` del producto.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np

MINIMO_DE_ESTRUCTURAS = 2
ESCALAS = (1.0, 100.0, 0.01)
#: Categorías del mmCIF que dicen de dónde viene el archivo (la detección del producto se apoya en ellas).
CATEGORIAS_DE_PROCEDENCIA = (
    "_entry.", "_exptl.", "_struct.", "_software.", "_audit_conform.", "_audit_author.", "_ma_model_list.",
    "_ma_qa_metric.", "_ma_qa_metric_global.", "_ma_data.", "_ma_protocol_step.", "_ma_software_group.",
    "_ma_target_entity.", "_entity.", "_struct_asym.", "_pdbx_database_status.",
)
MAX_FILAS_POR_CATEGORIA = 25
MAX_LINEAS_DE_CABECERA_PDB = 60


# ── Utilidades ───────────────────────────────────────────────────────


def sha256(ruta: Path | str) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def _decimales(texto: str) -> int:
    texto = texto.strip()
    if "e" in texto.lower() or "." not in texto:
        return 0
    return len(texto.split(".", 1)[1])


def _flotante(texto: str) -> float | None:
    try:
        valor = float(texto)
    except (TypeError, ValueError):
        return None
    return valor if math.isfinite(valor) else None


def _rangos(valores: np.ndarray) -> np.ndarray:
    """Rangos con empates promediados (para la ρ de Spearman sin depender de scipy)."""
    orden = np.argsort(valores, kind="mergesort")
    rangos = np.empty(len(valores), dtype=float)
    ordenados = valores[orden]
    i = 0
    while i < len(valores):
        j = i
        while j + 1 < len(valores) and ordenados[j + 1] == ordenados[i]:
            j += 1
        rangos[orden[i:j + 1]] = (i + j) / 2.0
        i = j + 1
    return rangos


def spearman(a: Iterable[float], b: Iterable[float]) -> float | None:
    # Redondeo antes de ordenar: la media de átomos idénticos puede diferir en un ulp y romper un empate.
    x, y = np.round(np.asarray(list(a), dtype=float), 6), np.round(np.asarray(list(b), dtype=float), 6)
    if len(x) != len(y) or len(x) < 3:
        return None
    rx, ry = _rangos(x), _rangos(y)
    if rx.std() == 0 or ry.std() == 0:
        return None
    return float(np.corrcoef(rx, ry)[0, 1])


# ── Lectura del archivo ──────────────────────────────────────────────


def _es_hidrogeno(elemento: str, nombre: str) -> bool:
    elemento = (elemento or "").strip().upper()
    if elemento:
        return elemento in ("H", "D")
    return nombre.strip().upper().startswith("H")


def _agrupar_residuos(atomos: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Residuos en el orden del archivo, con los factores B de sus átomos pesados."""
    residuos: list[dict[str, Any]] = []
    indice: dict[tuple, int] = {}
    for atomo in atomos:
        clave = atomo["residuo"]
        if clave not in indice:
            indice[clave] = len(residuos)
            residuos.append({"clave": clave, "nombre": atomo["comp"], "proteina": atomo["proteina"],
                             "cadena": atomo["cadena"], "label_seq": atomo.get("label_seq"),
                             "numero": atomo["numero"], "b": [], "b_ca": None})
        residuo = residuos[indice[clave]]
        residuo["b"].append(atomo["b"])
        if atomo["nombre"] == "CA" and residuo["b_ca"] is None:
            residuo["b_ca"] = atomo["b"]
    for residuo in residuos:
        residuo["b_media"] = float(np.mean(residuo["b"]))
        residuo["b_dispersion"] = float(max(residuo["b"]) - min(residuo["b"]))
    return residuos


def _valor_de_categoria(bloque, categoria: str) -> dict[str, list[str]]:
    import gemmi

    crudo = bloque.get_mmcif_category(categoria, raw=True)
    return {tag: [gemmi.cif.as_string(v) if v is not None else "?" for v in valores[:MAX_FILAS_POR_CATEGORIA]]
            for tag, valores in crudo.items()}


def leer_mmcif(ruta: Path | str) -> dict[str, Any]:
    """Lo que declara un mmCIF (procedencia, métricas ModelCIF) y el factor B de cada átomo, en el orden del archivo."""
    import gemmi

    bloque = gemmi.cif.read(str(ruta)).sole_block()
    categorias = sorted(bloque.get_mmcif_category_names())
    procedencia = {c: _valor_de_categoria(bloque, c) for c in CATEGORIAS_DE_PROCEDENCIA if c in categorias}

    # Qué cadenas son polímero, según el propio archivo (si lo declara).
    tipo_de_entidad = {gemmi.cif.as_string(f[0]): gemmi.cif.as_string(f[1]).lower()
                       for f in bloque.find("_entity.", ["id", "type"])}
    asym_polimero = {gemmi.cif.as_string(f[0]) for f in bloque.find("_struct_asym.", ["id", "entity_id"])
                     if tipo_de_entidad.get(gemmi.cif.as_string(f[1])) == "polymer"}
    criterio_polimero = "_entity.type" if asym_polimero else "group_PDB"

    tabla = bloque.find("_atom_site.", ["group_PDB", "label_atom_id", "label_comp_id", "label_asym_id",
                                         "?label_seq_id", "?auth_seq_id", "?auth_asym_id", "?type_symbol",
                                         "B_iso_or_equiv", "?pdbx_PDB_model_num", "?pdbx_PDB_ins_code"])
    atomos: list[dict[str, Any]] = []
    decimales_b = 0
    hidrogenos = 0
    primer_modelo = None
    for fila in tabla:
        texto = [gemmi.cif.as_string(fila[i]) if fila.has(i) else "" for i in range(len(fila))]
        modelo = texto[9] or "1"
        primer_modelo = primer_modelo or modelo
        if modelo != primer_modelo:
            continue
        if _es_hidrogeno(texto[7], texto[1]):
            hidrogenos += 1
            continue
        b = _flotante(texto[8])
        if b is None:
            continue
        decimales_b = max(decimales_b, _decimales(fila.str(8)))
        asym = texto[3]
        proteina = (asym in asym_polimero) if asym_polimero else texto[0].upper() == "ATOM"
        label_seq = int(texto[4]) if texto[4] not in ("", ".", "?") and texto[4].lstrip("-").isdigit() else None
        numero = texto[5] or texto[4]
        atomos.append({"nombre": texto[1], "comp": texto[2], "cadena": texto[6] or asym, "asym": asym,
                       "label_seq": label_seq, "numero": numero, "b": b, "proteina": proteina,
                       "residuo": (asym, label_seq if label_seq is not None else numero, texto[2], texto[10])})

    # Métricas ModelCIF: nombres y valores locales por métrica.
    metricas = [{"id": gemmi.cif.as_string(f[0]), "nombre": gemmi.cif.as_string(f[1]),
                 "tipo": gemmi.cif.as_string(f[2]) if f.has(2) else None,
                 "modo": gemmi.cif.as_string(f[3]) if f.has(3) else None}
                for f in bloque.find("_ma_qa_metric.", ["id", "name", "?type", "?mode"])]
    locales: dict[str, dict[tuple[str, int], float]] = {}
    decimales_local: dict[str, int] = {}
    for f in bloque.find("_ma_qa_metric_local.", ["label_asym_id", "label_seq_id", "metric_id", "metric_value"]):
        metrica = gemmi.cif.as_string(f[2])
        valor = _flotante(gemmi.cif.as_string(f[3]))
        seq = gemmi.cif.as_string(f[1])
        if valor is None or not seq.lstrip("-").isdigit():
            continue
        locales.setdefault(metrica, {})[(gemmi.cif.as_string(f[0]), int(seq))] = valor
        decimales_local[metrica] = max(decimales_local.get(metrica, 0), _decimales(f.str(3)))
    globales = [{"metrica": gemmi.cif.as_string(f[0]), "valor": gemmi.cif.as_string(f[1])}
                for f in bloque.find("_ma_qa_metric_global.", ["metric_id", "metric_value"])]

    return {"formato": "mmcif", "categorias": categorias, "procedencia": procedencia,
            "criterio_polimero": criterio_polimero, "atomos": atomos, "residuos": _agrupar_residuos(atomos),
            "decimales_b": decimales_b, "hidrogenos_omitidos": hidrogenos, "metricas": metricas,
            "locales": locales, "decimales_local": decimales_local, "globales": globales}


def leer_pdb(ruta: Path | str) -> dict[str, Any]:
    """La cabecera de un PDB clásico y el factor B de cada átomo pesado del primer modelo."""
    cabecera: list[str] = []
    atomos: list[dict[str, Any]] = []
    decimales_b = 0
    hidrogenos = 0
    en_atomos = False
    for linea in Path(ruta).read_text(encoding="utf-8", errors="replace").splitlines():
        registro = linea[:6].strip()
        if registro == "ENDMDL":
            break
        if registro not in ("ATOM", "HETATM"):
            if not en_atomos and len(cabecera) < MAX_LINEAS_DE_CABECERA_PDB:
                cabecera.append(linea.rstrip())
            continue
        en_atomos = True
        nombre, elemento = linea[12:16].strip(), linea[76:78].strip()
        if _es_hidrogeno(elemento, nombre):
            hidrogenos += 1
            continue
        crudo = linea[60:66]
        b = _flotante(crudo)
        if b is None:
            continue
        decimales_b = max(decimales_b, _decimales(crudo))
        cadena, numero, insercion, comp = linea[21:22].strip(), linea[22:26].strip(), linea[26:27].strip(), linea[17:20].strip()
        atomos.append({"nombre": nombre, "comp": comp, "cadena": cadena, "asym": cadena, "label_seq": None,
                       "numero": numero, "b": b, "proteina": registro == "ATOM",
                       "residuo": (cadena, numero, comp, insercion)})
    return {"formato": "pdb", "cabecera": cabecera, "criterio_polimero": "registro ATOM", "atomos": atomos,
            "residuos": _agrupar_residuos(atomos), "decimales_b": decimales_b, "hidrogenos_omitidos": hidrogenos,
            "metricas": [], "locales": {}, "decimales_local": {}, "globales": []}


def leer_estructura(ruta: Path | str) -> dict[str, Any]:
    return leer_pdb(ruta) if str(ruta).lower().endswith((".pdb", ".ent")) else leer_mmcif(ruta)


# ── Series que se comparan ───────────────────────────────────────────


def series_del_archivo(lectura: dict[str, Any]) -> dict[str, tuple[list[float], int]]:
    """Cada candidato a «el pLDDT del archivo», con los decimales con que está escrito."""
    proteina = [r for r in lectura["residuos"] if r["proteina"]]
    otros = [r for r in lectura["residuos"] if not r["proteina"]]
    d = lectura["decimales_b"]
    series: dict[str, tuple[list[float], int]] = {
        "b_media_por_residuo_proteina": ([r["b_media"] for r in proteina], d),
        "b_ca_por_residuo_proteina": ([r["b_ca"] if r["b_ca"] is not None else r["b"][0] for r in proteina], d),
        "b_por_atomo_proteina": ([a["b"] for a in lectura["atomos"] if a["proteina"]], d),
        "b_por_atomo_todo": ([a["b"] for a in lectura["atomos"]], d),
        # Un residuo de proteína = un token; cada átomo pesado de un ligando = un token (como tokeniza Boltz).
        "b_por_token": ([r["b_media"] for r in proteina] + [b for r in otros for b in r["b"]], d),
    }
    for metrica, valores in lectura["locales"].items():
        nombre = next((m["nombre"] for m in lectura["metricas"] if m["id"] == metrica), metrica)
        serie = [valores.get((r["clave"][0], r["label_seq"])) for r in proteina]
        if serie and all(v is not None for v in serie):
            series[f"ma_qa_metric_local[{nombre}]_proteina"] = (serie, lectura["decimales_local"].get(metrica, 0))
    return {k: v for k, v in series.items() if v[0]}


def series_del_modelo(valores: np.ndarray, lectura: dict[str, Any]) -> dict[str, list[float]]:
    plano = np.asarray(valores, dtype=float).reshape(-1)
    n_res = sum(1 for r in lectura["residuos"] if r["proteina"])
    n_atomos_proteina = sum(1 for a in lectura["atomos"] if a["proteina"])
    series = {"modelo": plano.tolist()}
    if len(plano) > n_res > 0:
        series["modelo_primeros_n_residuos"] = plano[:n_res].tolist()
    if len(plano) > n_atomos_proteina > 0:
        series["modelo_primeros_n_atomos_proteina"] = plano[:n_atomos_proteina].tolist()
    return series


def comparar(archivo: list[float], modelo: list[float], decimales: int) -> dict[str, Any]:
    """¿Reproduce el archivo la salida del modelo con una sola escala, hasta el redondeo con que escribe?"""
    if len(archivo) != len(modelo):
        return {"coincide": False, "motivo": "longitud", "n_archivo": len(archivo), "n_modelo": len(modelo)}
    a, m = np.asarray(archivo, dtype=float), np.asarray(modelo, dtype=float)
    tolerancia = 0.5 * 10.0 ** (-decimales) + 1e-5
    mejor: dict[str, Any] | None = None
    for escala in ESCALAS:
        dif = float(np.max(np.abs(a - m * escala)))
        if mejor is None or dif < mejor["max_dif"]:
            mejor = {"escala": escala, "max_dif": dif}
    pearson = float(np.corrcoef(a, m)[0, 1]) if len(a) > 2 and a.std() > 0 and m.std() > 0 else None
    return {"coincide": mejor["max_dif"] <= tolerancia, "escala": mejor["escala"], "max_dif": mejor["max_dif"],
            "tolerancia": tolerancia, "decimales": decimales, "n": len(a), "pearson": pearson}


def comparar_estructura(lectura: dict[str, Any], valores_modelo: np.ndarray | None) -> list[dict[str, Any]]:
    if valores_modelo is None:
        return []
    salida = []
    archivo = series_del_archivo(lectura)
    modelo = series_del_modelo(valores_modelo, lectura)
    for nombre_a, (serie_a, decimales) in archivo.items():
        for nombre_m, serie_m in modelo.items():
            if len(serie_a) != len(serie_m):
                continue
            salida.append({"serie_archivo": nombre_a, "serie_modelo": nombre_m,
                           **comparar(serie_a, serie_m, decimales)})
    return salida


def dispersion_dentro_del_residuo(lectura: dict[str, Any]) -> dict[str, Any]:
    """Si el factor B cambia de un átomo a otro del mismo residuo (pLDDT por átomo) o es uno por residuo."""
    proteina = [r for r in lectura["residuos"] if r["proteina"]]
    if not proteina:
        return {"residuos": 0}
    maxima = max(r["b_dispersion"] for r in proteina)
    tolerancia = 0.5 * 10.0 ** (-lectura["decimales_b"]) * 2 + 1e-5
    return {"residuos": len(proteina), "max_dispersion": maxima,
            "uno_por_residuo": maxima <= tolerancia,
            "residuos_con_dispersion": sum(1 for r in proteina if r["b_dispersion"] > tolerancia)}


def resumen_de_la_estructura(lectura: dict[str, Any]) -> dict[str, Any]:
    proteina = [r for r in lectura["residuos"] if r["proteina"]]
    otros = [r for r in lectura["residuos"] if not r["proteina"]]
    b = [a["b"] for a in lectura["atomos"]]
    return {
        "formato": lectura["formato"],
        "criterio_polimero": lectura["criterio_polimero"],
        "residuos_proteina": len(proteina),
        "residuos_no_proteina": [{"nombre": r["nombre"], "cadena": r["cadena"], "numero": r["numero"],
                                  "atomos": len(r["b"]), "b_min": min(r["b"]), "b_max": max(r["b"])} for r in otros],
        "atomos_pesados": len(b),
        "hidrogenos_omitidos": lectura["hidrogenos_omitidos"],
        "b_min": min(b) if b else None, "b_max": max(b) if b else None,
        "decimales_b": lectura["decimales_b"],
        "dentro_del_residuo": dispersion_dentro_del_residuo(lectura),
        "metricas_modelcif": lectura["metricas"],
        "metricas_globales": lectura["globales"],
        "categorias": lectura.get("categorias"),
        "procedencia": lectura.get("procedencia"),
        "cabecera_pdb": lectura.get("cabecera"),
    }


# ── Veredicto ────────────────────────────────────────────────────────


def escala_del_archivo(escala_sobre_modelo: float, max_modelo: float) -> str:
    modelo_0_1 = max_modelo <= 1.0 + 1e-6
    valor = (1.0 if modelo_0_1 else 100.0) * escala_sobre_modelo
    return "0-1" if abs(valor - 1.0) < 1e-9 else "0-100" if abs(valor - 100.0) < 1e-9 else f"×{valor:g}"


def veredicto(estructuras: list[dict[str, Any]]) -> dict[str, Any]:
    """``estructuras``: [{herramienta, formato, archivo, max_modelo, comparaciones: [...]}, ...] de UN grupo."""
    terminadas = [e for e in estructuras if e.get("comparaciones") is not None]
    candidatos = sorted({c["serie_archivo"] for e in terminadas for c in e["comparaciones"]})
    por_serie: dict[str, Any] = {}
    for serie in candidatos:
        escalas, difs, cubiertas = set(), [], 0
        for e in terminadas:
            buenas = [c for c in e["comparaciones"] if c["serie_archivo"] == serie and c["coincide"]]
            if buenas:
                cubiertas += 1
                mejor = min(buenas, key=lambda c: c["max_dif"])
                escalas.add(escala_del_archivo(mejor["escala"], e["max_modelo"]))
                difs.append(mejor["max_dif"])
        verificada = (len(terminadas) >= MINIMO_DE_ESTRUCTURAS and cubiertas == len(terminadas)
                      and len(escalas) == 1)
        por_serie[serie] = {"verificada": verificada, "estructuras_que_coinciden": cubiertas,
                            "estructuras": len(terminadas), "escala_en_el_archivo": sorted(escalas),
                            "max_dif": max(difs) if difs else None}
    verificadas = [s for s, v in por_serie.items() if v["verificada"]]
    if len(terminadas) < MINIMO_DE_ESTRUCTURAS:
        motivo = f"sólo {len(terminadas)} estructura(s) terminada(s); el criterio pide {MINIMO_DE_ESTRUCTURAS}"
    elif not verificadas:
        motivo = "ningún campo del archivo reproduce el pLDDT del modelo en todas las estructuras"
    else:
        motivo = None
    return {"estructuras": len(terminadas), "series_verificadas": verificadas, "motivo_si_no": motivo,
            "por_serie": por_serie}


# ── Recorrido de los resultados del cuaderno ─────────────────────────


def _modelo_boltz(cif: Path) -> tuple[np.ndarray | None, dict[str, Any] | None]:
    m = re.match(r"(.+)_model_(\d+)\.(cif|pdb)$", cif.name)
    if not m:
        return None, None
    nombre, k = m.group(1), m.group(2)
    npz = cif.parent / f"plddt_{nombre}_model_{k}.npz"
    conf = cif.parent / f"confidence_{nombre}_model_{k}.json"
    valores = None
    if npz.exists():
        with np.load(npz) as datos:
            clave = "plddt" if "plddt" in datos.files else datos.files[0]
            valores = np.asarray(datos[clave], dtype=float)
    confianza = json.loads(conf.read_text(encoding="utf-8")) if conf.exists() else None
    return valores, confianza


def _modelo_chai(cif: Path, corrida: Path) -> tuple[np.ndarray | None, dict[str, Any] | None]:
    resumen_ruta = corrida / "resumen_chai.json"
    npz = corrida / "salida_del_modelo.npz"
    if not (resumen_ruta.exists() and npz.exists()):
        return None, None
    resumen = json.loads(resumen_ruta.read_text(encoding="utf-8"))
    rutas = [Path(p).name for p in resumen.get("cif_paths") or []]
    if cif.name not in rutas:
        return None, None
    i = rutas.index(cif.name)
    with np.load(npz) as datos:
        if "plddt" not in datos.files:
            return None, None
        plddt = np.asarray(datos["plddt"], dtype=float)
    valores = plddt[i] if plddt.ndim >= 2 else plddt
    puntuaciones = corrida / cif.name.replace("pred.", "scores.").replace(".cif", ".npz")
    extra = None
    if puntuaciones.exists():
        with np.load(puntuaciones) as datos:
            extra = {k: _a_lista(datos[k]) for k in datos.files}
    return valores, extra


def _a_lista(valor) -> Any:
    """Un array del npz como lista JSON. Sólo se redondea lo que es número real: Chai-1 guarda también booleanos."""
    arreglo = np.asarray(valor)
    return arreglo.round(4).tolist() if np.issubdtype(arreglo.dtype, np.floating) else arreglo.tolist()


def analizar_resultados(raiz: Path | str) -> dict[str, Any]:
    """Recorre ``resultados/boltz/*`` y ``resultados/chai/*`` y aplica el criterio por grupo."""
    raiz = Path(raiz)
    estructuras: list[dict[str, Any]] = []
    for herramienta in ("boltz", "chai"):
        for corrida in sorted((raiz / herramienta).glob("*")) if (raiz / herramienta).exists() else []:
            if not corrida.is_dir() or "__fallido_" in corrida.name:
                continue
            caso, _, formato_pedido = corrida.name.partition("__")
            for ruta in sorted(list(corrida.rglob("*.cif")) + list(corrida.rglob("*.pdb"))):
                if herramienta == "boltz" and "_model_" not in ruta.name:
                    continue
                if herramienta == "chai" and not ruta.name.startswith("pred."):
                    continue
                lectura = leer_estructura(ruta)
                valores, confianza = (_modelo_boltz(ruta) if herramienta == "boltz" else _modelo_chai(ruta, corrida))
                estructuras.append({
                    "herramienta": herramienta, "caso": caso, "formato": lectura["formato"],
                    "archivo": str(ruta.relative_to(raiz)).replace("\\", "/"), "sha256": sha256(ruta),
                    "max_modelo": float(np.max(valores)) if valores is not None else None,
                    "media_modelo": float(np.mean(valores)) if valores is not None else None,
                    "n_modelo": int(np.asarray(valores).size) if valores is not None else None,
                    "confianza_declarada_por_el_modelo": confianza,
                    "resumen": resumen_de_la_estructura(lectura),
                    "comparaciones": comparar_estructura(lectura, valores) if valores is not None else None,
                    "_lectura": lectura,
                })
    grupos: dict[str, Any] = {}
    for clave in sorted({(e["herramienta"], e["formato"]) for e in estructuras}):
        grupo = [e for e in estructuras if (e["herramienta"], e["formato"]) == clave]
        grupos[f"{clave[0]}/{clave[1]}"] = veredicto(grupo)

    control = _control_alphafold_db(raiz, estructuras, grupos)
    for e in estructuras:
        e.pop("_lectura", None)
    return {"criterio": {"minimo_de_estructuras": MINIMO_DE_ESTRUCTURAS, "escalas": list(ESCALAS),
                         "tolerancia": "0,5 × 10^-decimales escritos (+1e-5)"},
            "grupos": grupos, "control_alphafold_db": control, "estructuras": estructuras}


def _control_alphafold_db(raiz: Path, estructuras: list[dict[str, Any]], grupos: dict[str, Any]) -> list[dict]:
    """ρ de Spearman entre el pLDDT de cada predicción de NET y el de AlphaFold DB, residuo a residuo."""
    af = next(iter(sorted((raiz / "referencias").glob("AF-*.cif"))), None) if (raiz / "referencias").exists() else None
    if af is None:
        return []
    lectura_af = leer_mmcif(af)
    plddt_af = next((v for m, v in lectura_af["locales"].items()
                     if any(x["id"] == m and x["nombre"].lower() == "plddt" for x in lectura_af["metricas"])), None)
    if not plddt_af:
        return []
    por_numero = {seq: v for (_, seq), v in plddt_af.items()}
    salida = []
    for e in estructuras:
        if e["caso"] != "net_apo":
            continue
        lectura = e["_lectura"]
        proteina = [r for r in lectura["residuos"] if r["proteina"]]
        numeros = [r["label_seq"] if r["label_seq"] is not None else int(r["numero"]) for r in proteina]
        comunes = [(r["b_media"], por_numero[n]) for r, n in zip(proteina, numeros) if n in por_numero]
        salida.append({"archivo": e["archivo"], "residuos_comunes": len(comunes),
                       "spearman_b_media_vs_afdb": spearman([c[0] for c in comunes], [c[1] for c in comunes]),
                       "referencia": af.name, "referencia_sha256": sha256(af)})
    return salida


def informe_markdown(analisis: dict[str, Any], procedencia: dict[str, Any] | None = None) -> str:
    lineas = ["# Convención del pLDDT de Boltz y Chai-1 — resultado", ""]
    if procedencia:
        lineas += [f"- GPU: {procedencia.get('gpu', {}).get('nombre')}",
                   f"- Boltz: {procedencia.get('versiones', {}).get('boltz')} · Chai-1: "
                   f"{procedencia.get('versiones', {}).get('chai_lab')}", ""]
    lineas += ["| Grupo | Estructuras | Campo verificado | Escala en el archivo | Motivo si no |",
               "|---|---:|---|---|---|"]
    for grupo, v in analisis["grupos"].items():
        series = v["series_verificadas"]
        escalas = sorted({e for s in series for e in v["por_serie"][s]["escala_en_el_archivo"]})
        lineas.append(f"| {grupo} | {v['estructuras']} | {', '.join(series) or '—'} | {', '.join(escalas) or '—'} | "
                      f"{v['motivo_si_no'] or ''} |")
    lineas += ["", "## Lo que declara cada archivo", ""]
    for e in analisis["estructuras"]:
        r = e["resumen"]
        lineas.append(f"### {e['archivo']}")
        lineas.append(f"- átomos pesados {r['atomos_pesados']}, residuos de proteína {r['residuos_proteina']}, "
                      f"B {r['b_min']}–{r['b_max']} con {r['decimales_b']} decimales; "
                      f"uno por residuo: {r['dentro_del_residuo'].get('uno_por_residuo')}")
        if r.get("metricas_modelcif"):
            lineas.append(f"- métricas ModelCIF: {', '.join(m['nombre'] for m in r['metricas_modelcif'])}")
        if r.get("procedencia"):
            for categoria in ("_exptl.", "_struct.", "_software.", "_audit_conform.", "_entry."):
                if categoria in r["procedencia"]:
                    lineas.append(f"- `{categoria}` {json.dumps(r['procedencia'][categoria], ensure_ascii=False)[:300]}")
        if r.get("cabecera_pdb") is not None:
            lineas.append(f"- cabecera PDB: {json.dumps(r['cabecera_pdb'][:8], ensure_ascii=False)[:400]}")
        lineas.append("")
    if analisis.get("control_alphafold_db"):
        lineas += ["## Control de sentido contra AlphaFold DB (NET)", ""]
        for c in analisis["control_alphafold_db"]:
            rho = c["spearman_b_media_vs_afdb"]
            lineas.append(f"- {c['archivo']}: ρ = {rho:.3f} sobre {c['residuos_comunes']} residuos" if rho is not None
                          else f"- {c['archivo']}: sin ρ")
    return "\n".join(lineas) + "\n"

In [ ]:
%%writefile /content/plddt_boltz_chai/ejecutar_chai.py
"""Corre Chai-1 sobre un FASTA y guarda, además de sus archivos, los tensores que devuelve ``run_inference``.

Se ejecuta DENTRO del entorno virtual de Chai-1 (``/content/venvs/chai/bin/python``), no en el del cuaderno: así sus
dependencias no chocan con las de Boltz. El tensor ``plddt`` que se guarda en ``salida_del_modelo.npz`` es la
referencia contra la que ``analisis_plddt.py`` compara el factor B del CIF.

Sólo pasa a ``run_inference`` los argumentos que la versión instalada acepta y deja escrito cuáles omitió: el registro
no puede decir que se pidió algo que la función no recibió.
"""

from __future__ import annotations

import argparse
import dataclasses
import inspect
import json
import sys
import time
from importlib import metadata
from pathlib import Path


def _serializable(valor, profundidad: int = 0):
    import numpy as np
    import torch

    if profundidad > 6:
        return repr(valor)[:200]
    if torch.is_tensor(valor):
        valor = valor.detach().float().cpu().numpy()
    if isinstance(valor, np.ndarray):
        return valor.round(6).tolist() if valor.size <= 2000 else {"forma": list(valor.shape)}
    if isinstance(valor, (np.floating, np.integer)):
        return valor.item()
    if isinstance(valor, Path):
        return str(valor)
    if dataclasses.is_dataclass(valor) and not isinstance(valor, type):
        return {f.name: _serializable(getattr(valor, f.name), profundidad + 1) for f in dataclasses.fields(valor)}
    if isinstance(valor, dict):
        return {str(k): _serializable(v, profundidad + 1) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [_serializable(v, profundidad + 1) for v in valor]
    if isinstance(valor, (str, int, float, bool)) or valor is None:
        return valor
    return repr(valor)[:200]


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--fasta", required=True)
    parser.add_argument("--salida", required=True, help="directorio nuevo y vacío")
    parser.add_argument("--semilla", type=int, default=42)
    parser.add_argument("--msa-servidor", type=int, choices=(0, 1), default=1)
    parser.add_argument("--muestras", type=int, default=5)
    args = parser.parse_args()

    import numpy as np
    import torch
    from chai_lab.chai1 import run_inference

    salida = Path(args.salida)
    salida.mkdir(parents=True, exist_ok=False)
    resumen: dict = {
        "chai_lab": metadata.version("chai_lab"), "torch": torch.__version__, "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "bf16": bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
        "python": sys.version.split()[0],
    }
    firma = inspect.signature(run_inference).parameters
    deseados = {
        "fasta_file": Path(args.fasta), "output_dir": salida, "num_trunk_recycles": 3,
        "num_diffn_timesteps": 200, "num_diffn_samples": args.muestras, "seed": args.semilla,
        "device": "cuda:0", "use_esm_embeddings": True, "use_msa_server": bool(args.msa_servidor),
    }
    kwargs = {k: v for k, v in deseados.items() if k in firma}
    resumen["argumentos_pasados"] = {k: str(v) for k, v in kwargs.items()}
    resumen["argumentos_no_aceptados_por_esta_version"] = sorted(set(deseados) - set(kwargs))
    resumen["firma_de_run_inference"] = [str(p) for p in firma.values()]
    if args.msa_servidor and "use_msa_server" not in firma:
        resumen["error"] = "esta versión de chai_lab no acepta use_msa_server: no se corre sin decirlo"
        (salida / "resumen_chai.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
        return 2

    t0 = time.time()
    try:
        candidatos = run_inference(**kwargs)
    except Exception as exc:  # se registra (memoria, bf16…) y el cuaderno lo cuenta como «no terminó»
        resumen["segundos"] = round(time.time() - t0, 1)
        resumen["error"] = f"{type(exc).__name__}: {str(exc)[:2000]}"
        (salida / "resumen_chai.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
        raise
    resumen["segundos"] = round(time.time() - t0, 1)

    tensores, otros = {}, {}
    campos = ([f.name for f in dataclasses.fields(candidatos)] if dataclasses.is_dataclass(candidatos)
              else [n for n in vars(candidatos)])
    for nombre in campos:
        valor = getattr(candidatos, nombre)
        if torch.is_tensor(valor):
            tensores[nombre] = valor.detach().float().cpu().numpy()
        else:
            otros[nombre] = _serializable(valor)
    np.savez_compressed(salida / "salida_del_modelo.npz", **tensores)
    resumen["tensores"] = {k: list(v.shape) for k, v in tensores.items()}
    resumen["cif_paths"] = [str(p) for p in getattr(candidatos, "cif_paths", [])]
    resumen["otros_campos"] = otros
    (salida / "resumen_chai.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({k: resumen[k] for k in ("chai_lab", "gpu", "bf16", "segundos", "tensores")}, ensure_ascii=False))
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# ── Secuencias de UniProt y la referencia de AlphaFold DB ──
import urllib.request


def descargar(url, destino=None):
    peticion = urllib.request.Request(url, headers={"User-Agent": "convencion-plddt/1.0"})
    with urllib.request.urlopen(peticion, timeout=120) as resp:
        datos = resp.read()
        cabeceras = {k: resp.headers.get(k) for k in ("X-UniProt-Release", "X-UniProt-Release-Date", "Last-Modified")}
    if destino:
        Path(destino).write_bytes(datos)
    return datos, cabeceras


secuencias = cargar_procedencia().get("secuencias", {})
for acc in sorted({c["uniprot"] for c in CASOS}):
    if acc in secuencias:
        continue
    datos, cab = descargar(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta")
    cabecera, *resto = datos.decode().strip().splitlines()
    seq = "".join(resto).strip()
    if not seq or not set(seq) <= set("ACDEFGHIKLMNPQRSTVWYXUO"):
        raise ValueError(f"secuencia inesperada para {acc}: {cabecera}")
    secuencias[acc] = {"cabecera": cabecera, "longitud": len(seq), "sha256": hashlib.sha256(seq.encode()).hexdigest(),
                       "version_uniprot": cab["X-UniProt-Release"], "fecha_version": cab["X-UniProt-Release-Date"],
                       "descargada": ahora(), "secuencia": seq}
    print(acc, len(seq), cabecera[:90])

ref_dir = RES / "referencias"
ref_dir.mkdir(exist_ok=True)
af_ruta = ref_dir / Path(REFERENCIA_AFDB["url"]).name
referencia = {"url": REFERENCIA_AFDB["url"]}
try:
    if not af_ruta.exists():
        _, cab = descargar(REFERENCIA_AFDB["url"], af_ruta)
        referencia["last_modified"] = cab["Last-Modified"]
    h = hashlib.sha256(af_ruta.read_bytes()).hexdigest()
    referencia.update(sha256=h, igual_que_el_del_producto=(h == REFERENCIA_AFDB["sha256_en_el_producto"]))
except Exception as exc:  # el control de sentido es opcional: su falta se registra y no para nada
    referencia["error"] = f"{type(exc).__name__}: {exc}"
print(referencia)
guardar_procedencia(secuencias=secuencias, referencia_afdb_descargada=referencia)
respaldar()

In [ ]:
# ── Instalación: sólo lo que tiene corridas pendientes, cada herramienta en su entorno virtual (sus dependencias chocan) ──
PENDIENTES = pendientes()
A_INSTALAR = [h for h in ("boltz", "chai") if PENDIENTES[h]]
print("Corridas pendientes:", PENDIENTES, "→ se instala:", A_INSTALAR or "nada")
sh([sys.executable, "-m", "pip", "install", "-q", "uv", "gemmi"], check=True)
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
PY = {"boltz": VENVS / "boltz" / "bin" / "python", "chai": VENVS / "chai" / "bin" / "python"}
DIST = {"boltz": "boltz", "chai": "chai_lab"}
PAQUETES = {"boltz": "boltz[cuda]" + (f"=={VERSION_BOLTZ}" if VERSION_BOLTZ else ""),
            "chai": "chai_lab" + (f"=={VERSION_CHAI}" if VERSION_CHAI else "")}

INFO = """
import json, sys
from importlib import metadata
import torch
d = metadata.distribution(sys.argv[1])
licencias = [f for f in (d.files or []) if "LICEN" in str(f).upper() or "COPYING" in str(f).upper()]
print(json.dumps({
    "version": d.version,
    "licencia_metadata": d.metadata.get("License"),
    "licencia_expresion": d.metadata.get("License-Expression"),
    "clasificadores_de_licencia": [c for c in (d.metadata.get_all("Classifier") or []) if "License" in c],
    "archivos_de_licencia": {str(f): d.locate_file(f).read_text(errors="replace")[:20000] for f in licencias},
    "torch": torch.__version__, "cuda": torch.version.cuda, "python": sys.version.split()[0],
}))
"""

# Chai-1 guarda el alineamiento del servidor MSA en parquet. En la primera ejecución pandas no pudo importar pyarrow
# (instalado, 26.0.1) y resumió el error como «falta pyarrow». La sonda hace lo mismo que Chai-1 y, si falla, imprime el
# error real de cada importación.
SONDA_PARQUET = """
import os, sys, tempfile, traceback
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
try:
    import numpy, pandas
    ruta = os.path.join(tempfile.mkdtemp(), "sonda.parquet")
    pandas.DataFrame({"a": [1, 2]}).to_parquet(ruta)
    assert pandas.read_parquet(ruta)["a"].tolist() == [1, 2]
    print("parquet funciona con pandas", pandas.__version__, "y numpy", numpy.__version__)
except Exception:
    traceback.print_exc()
    for modulo in ("pyarrow", "pyarrow.parquet", "fastparquet"):
        try:
            __import__(modulo)
            print(modulo, "se importa")
        except Exception:
            print("--- error real al importar", modulo)
            traceback.print_exc()
    sys.exit(1)
"""
#: Lo que se prueba, en orden, si la sonda falla. Cada intento queda registrado con su salida.
ARREGLOS_PARQUET = ("pyarrow<22", "pyarrow<20", "pyarrow<18", "fastparquet")


def sondear_parquet():
    r, _ = sh([str(PY["chai"]), "-c", SONDA_PARQUET])
    return r.returncode == 0, ((r.stdout or "") + (r.stderr or "")).strip()[-6000:]


previa = cargar_procedencia()
instalacion = dict(previa.get("instalacion", {}))
historial = list(previa.get("instalaciones_anteriores", []))
versiones = dict(previa.get("versiones", {}))
(RES / "licencias").mkdir(exist_ok=True)
instalacion["uv"] = sh([UV, "--version"])[0].stdout.strip()
for nombre in A_INSTALAR:
    dist = DIST[nombre]
    if isinstance(instalacion.get(nombre), dict):
        historial.append({"herramienta": nombre, "apartada": ahora(), **instalacion[nombre]})
    for viejo in (RES / f"instalacion_{nombre}.log", RES / f"pip_freeze_{nombre}.txt"):
        if viejo.exists():
            apartar(viejo, "anterior")
    if not PY[nombre].exists():
        sh([UV, "venv", str(VENVS / nombre), "--python", "3.11"], check=True)
    r, s = sh([UV, "pip", "install", "--python", str(PY[nombre]), PAQUETES[nombre]],
              log=RES / f"instalacion_{nombre}.log")
    if r.returncode != 0:
        raise RuntimeError(f"No se instaló {PAQUETES[nombre]}: mira {RES / f'instalacion_{nombre}.log'}")
    registro = {"paquete_pedido": PAQUETES[nombre], "segundos": s, "instalado": ahora()}
    if nombre == "chai":
        CHAI_LISTO, salida = sondear_parquet()
        intentos = [{"cambio": None, "funciona": CHAI_LISTO, "salida": salida}]
        print("Sonda de parquet en el entorno de Chai-1:\n" + salida[-2500:])
        for arreglo in ARREGLOS_PARQUET:
            if CHAI_LISTO:
                break
            log_arreglo = RES / f"instalacion_chai_arreglo_{arreglo.replace('<', '_menor_que_')}.log"
            if log_arreglo.exists():
                apartar(log_arreglo, "anterior")
            r, _ = sh([UV, "pip", "install", "--python", str(PY["chai"]), arreglo], log=log_arreglo)
            CHAI_LISTO, salida = sondear_parquet()
            intentos.append({"cambio": arreglo, "codigo_instalacion": r.returncode, "funciona": CHAI_LISTO,
                             "salida": salida})
            print(f"Con {arreglo}: {'funciona' if CHAI_LISTO else 'sigue sin funcionar'}")
        registro["parquet"] = {"funciona": CHAI_LISTO, "intentos": intentos}
        if not CHAI_LISTO:
            print("Parquet no funciona en el entorno de Chai-1 con ningún arreglo: Chai-1 no se lanzará.")
    congelado, _ = sh([UV, "pip", "freeze", "--python", str(PY[nombre])], check=True)
    (RES / f"pip_freeze_{nombre}.txt").write_text(congelado.stdout, encoding="utf-8")
    r, _ = sh([str(PY[nombre]), "-c", INFO, dist], check=True)
    info = json.loads(r.stdout.strip().splitlines()[-1])
    for archivo, texto in info.pop("archivos_de_licencia").items():
        (RES / "licencias" / f"{dist}__{Path(archivo).name}").write_text(texto, encoding="utf-8")
    instalacion[nombre] = {**registro, **info}
    versiones[dist] = info["version"]
print("Versiones registradas:", versiones)
guardar_procedencia(instalacion=instalacion, instalaciones_anteriores=historial, versiones=versiones)
respaldar()

In [ ]:
# ── Entradas: YAML de Boltz y FASTA de Chai-1 ──
ENT = RES / "entradas"
ENT.mkdir(exist_ok=True)
secuencias = cargar_procedencia()["secuencias"]
entradas = {}
for caso in CASOS:
    seq = secuencias[caso["uniprot"]]["secuencia"]
    proteina = {"id": "A", "sequence": seq}
    if not USAR_SERVIDOR_MSA:
        proteina["msa"] = "empty"  # una sola secuencia, sin alineamiento
    cadenas = [{"protein": proteina}]
    if caso["ligando"]:
        cadenas.append({"ligand": {"id": "B", "smiles": caso["ligando"]["smiles"]}})
    yaml_ruta = ENT / f"{caso['id']}.yaml"  # JSON es YAML válido: sin dependencias ni problemas de comillas
    yaml_ruta.write_text(json.dumps({"version": 1, "sequences": cadenas}, indent=2), encoding="utf-8")
    fasta = f">protein|name={caso['id']}_A\n{seq}\n"
    if caso["ligando"]:
        fasta += f">ligand|name={caso['id']}_B\n{caso['ligando']['smiles']}\n"
    fasta_ruta = ENT / f"{caso['id']}.fasta"
    fasta_ruta.write_text(fasta, encoding="utf-8")
    entradas[caso["id"]] = {
        "boltz": yaml_ruta.name, "sha256_boltz": hashlib.sha256(yaml_ruta.read_bytes()).hexdigest(),
        "chai": fasta_ruta.name, "sha256_chai": hashlib.sha256(fasta_ruta.read_bytes()).hexdigest(),
    }
guardar_procedencia(entradas=entradas)
respaldar()
print(json.dumps(entradas, indent=1))

In [ ]:
# ── Boltz: una corrida por caso y formato. Las terminadas no se repiten; las fallidas se apartan, no se borran ──
PENDIENTES_BOLTZ = pendientes()["boltz"]
if not PENDIENTES_BOLTZ:
    print("Boltz: todas las corridas están hechas; no se repite ninguna.")
else:
    BOLTZ = str(VENVS / "boltz" / "bin" / "boltz")
    AYUDA_BOLTZ = sh([BOLTZ, "predict", "--help"], check=True)[0].stdout
    (RES / "boltz").mkdir(exist_ok=True)
    corridas = cargar_procedencia().get("corridas_boltz", {})
    anteriores = cargar_procedencia().get("corridas_boltz_anteriores", [])
    for caso in CASOS:
        for formato in FORMATOS_BOLTZ[caso["id"]]:
            clave = f"{caso['id']}__{formato}"
            if clave not in PENDIENTES_BOLTZ:
                print("ya hecha:", clave)
                continue
            destino = RES / "boltz" / clave
            if destino.exists():  # su boltz.log va dentro y se aparta con ella
                apartada = apartar(destino)
                anteriores.append({"clave": clave, "apartada_como": apartada.name, **corridas.get(clave, {})})
            orden = [BOLTZ, "predict", str(ENT / f"{caso['id']}.yaml"), "--out_dir", str(destino),
                     "--cache", str(CACHE / "boltz"), "--output_format", formato, "--diffusion_samples", "1",
                     "--recycling_steps", "3", "--sampling_steps", "200", "--accelerator", "gpu", "--devices", "1"]
            pedidas = [("--seed", str(SEMILLA)), ("--write_full_pae", None), ("--override", None)]
            if USAR_SERVIDOR_MSA:
                pedidas.append(("--use_msa_server", None))
            if CAPACIDAD < 8.0:
                pedidas.append(("--no_kernels", None))
            no_aceptadas = []
            for bandera, valor in pedidas:
                if bandera in AYUDA_BOLTZ:
                    orden += [bandera] + ([valor] if valor is not None else [])
                else:
                    no_aceptadas.append(bandera)
            if USAR_SERVIDOR_MSA and "--use_msa_server" in no_aceptadas:
                raise RuntimeError("Esta versión de Boltz no acepta --use_msa_server: no se corre sin decirlo.")
            destino.mkdir(parents=True)
            r, s = sh(orden, log=destino / "boltz.log")
            corridas[clave] = {"orden": orden, "banderas_no_aceptadas": no_aceptadas, "codigo": r.returncode,
                               "segundos": s, "fin": ahora()}
            print(f"{clave}: código {r.returncode} en {s} s" + (f" (sin {no_aceptadas})" if no_aceptadas else ""))
            if r.returncode != 0:
                print("".join(open(destino / "boltz.log", encoding="utf-8", errors="replace").readlines()[-25:]))
            guardar_procedencia(corridas_boltz=corridas, corridas_boltz_anteriores=anteriores)
            respaldar()

In [ ]:
# ── Chai-1: una corrida por caso, con ejecutar_chai.py dentro de su propio entorno ──
PENDIENTES_CHAI = pendientes()["chai"]
if not PENDIENTES_CHAI:
    print("Chai-1: todas las corridas están hechas; no se repite ninguna.")
elif CHAI_LISTO is False:
    print("Chai-1 no se lanza: pandas no puede escribir parquet en su entorno. "
          "Los intentos están en procedencia.json → instalacion → chai → parquet.")
else:
    CHAI_PY = str(VENVS / "chai" / "bin" / "python")
    ENTORNO_CHAI = {**ENTORNO_LIMPIO, "CHAI_DOWNLOADS_DIR": str(CACHE / "chai")}
    (RES / "chai").mkdir(exist_ok=True)
    corridas = cargar_procedencia().get("corridas_chai", {})
    anteriores = cargar_procedencia().get("corridas_chai_anteriores", [])
    for caso in CASOS:
        clave = f"{caso['id']}__mmcif"
        if clave not in PENDIENTES_CHAI:
            print("ya hecha:", clave)
            continue
        destino = RES / "chai" / clave
        log = RES / "chai" / f"{clave}.log"  # fuera de destino: Chai-1 exige un directorio vacío
        if destino.exists() or log.exists():
            if destino.exists():
                apartada = apartar(destino)
            else:
                apartada = destino.with_name(f"{clave}__fallido_{sello()}")
                apartada.mkdir()
            if log.exists():
                log.rename(apartada / "chai.log")  # el registro de la corrida fallida viaja con ella
            anteriores.append({"clave": clave, "apartada_como": apartada.name, **corridas.get(clave, {})})
        orden = [CHAI_PY, str(RAIZ / "ejecutar_chai.py"), "--fasta", str(ENT / f"{caso['id']}.fasta"),
                 "--salida", str(destino), "--semilla", str(SEMILLA), "--msa-servidor", "1" if USAR_SERVIDOR_MSA else "0",
                 "--muestras", str(MUESTRAS_CHAI)]
        r, s = sh(orden, log=log, env=ENTORNO_CHAI)
        corridas[clave] = {"orden": orden, "codigo": r.returncode, "segundos": s, "fin": ahora()}
        print(f"{clave}: código {r.returncode} en {s} s")
        if r.returncode != 0:
            print("".join(open(log, encoding="utf-8", errors="replace").readlines()[-25:]))
        guardar_procedencia(corridas_chai=corridas, corridas_chai_anteriores=anteriores)
        respaldar()

In [ ]:
# ── Huella de los pesos que se usaron (SHA-256 de cada archivo de 1 MB o más) ──
def huella_de_directorio(raiz, minimo=1 << 20):
    grandes, pequenos, total = {}, 0, 0
    for p in sorted(Path(raiz).rglob("*")):
        if not p.is_file():
            continue
        tam = p.stat().st_size
        total += tam
        if tam < minimo:
            pequenos += 1
            continue
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for bloque in iter(lambda: f.read(1 << 24), b""):
                h.update(bloque)
        grandes[str(p.relative_to(raiz))] = {"bytes": tam, "sha256": h.hexdigest()}
    return {"archivos_grandes": grandes, "archivos_pequenos_sin_hash": pequenos, "bytes_totales": total}


directorios = {"boltz": CACHE / "boltz", "chai": CACHE / "chai",
               "huggingface": Path.home() / ".cache" / "huggingface", "torch_hub": Path.home() / ".cache" / "torch"}
previa = cargar_procedencia()
pesos = dict(previa.get("pesos", {}))  # lo que esta ejecución no descargó conserva la huella de la que sí lo hizo
historial_pesos = list(previa.get("pesos_anteriores", []))
medidos_ahora = set()
for n, d in directorios.items():
    if not (d.exists() and any(p.is_file() for p in d.rglob("*"))):
        continue
    if n in pesos:
        historial_pesos.append({"directorio": n, "apartada": ahora(), **pesos[n]})
    pesos[n] = {**huella_de_directorio(d), "medido": ahora()}
    medidos_ahora.add(n)
for n, v in pesos.items():
    print(n, len(v["archivos_grandes"]), "archivos grandes,", round(v["bytes_totales"] / 1e9, 2), "GB",
          "(medido en esta ejecución)" if n in medidos_ahora else "(de una ejecución anterior)")
guardar_procedencia(pesos=pesos, pesos_anteriores=historial_pesos)
respaldar()

In [ ]:
# ── Análisis: cada campo del archivo frente a la salida del modelo, con el criterio fijado arriba ──
import importlib

sys.path.insert(0, str(RAIZ))
import analisis_plddt as A

importlib.reload(A)
analisis = A.analizar_resultados(RES)
(RES / "analisis.json").write_text(json.dumps(analisis, ensure_ascii=False, indent=1), encoding="utf-8")
ejecuciones = cargar_procedencia().get("ejecuciones", [])
if ejecuciones:
    ejecuciones[-1]["fin"] = ahora()
procedencia = guardar_procedencia(fin=ahora(), ejecuciones=ejecuciones,
                                  sha256_analisis_plddt=A.sha256(RAIZ / "analisis_plddt.py"),
                                  sha256_ejecutar_chai=A.sha256(RAIZ / "ejecutar_chai.py"))
informe = A.informe_markdown(analisis, procedencia)
(RES / "CONVENCION.md").write_text(informe, encoding="utf-8")
respaldar()

from IPython.display import Markdown, display

display(Markdown(informe))

In [ ]:
# ── Descarga: un zip con todo (entradas, salidas, procedencia, análisis) ──
# Con la hora en el nombre: una segunda ejecución del mismo día no pisa el zip de la primera en Drive.
zip_ruta = Path(shutil.make_archive(f"/content/plddt_boltz_chai_{sello()}", "zip", RES))
zip_sha = hashlib.sha256(zip_ruta.read_bytes()).hexdigest()
print(zip_ruta.name, round(zip_ruta.stat().st_size / 1e6, 1), "MB  SHA-256", zip_sha)
if GUARDAR_EN_DRIVE and Path("/content/drive/MyDrive").exists():
    shutil.copy(zip_ruta, DRIVE.parent / zip_ruta.name)
    print("Copia en Drive:", DRIVE.parent / zip_ruta.name)
from google.colab import files

files.download(str(zip_ruta))
print("Hecho. Desconecta el entorno: Entorno de ejecución → Desconectar y eliminar entorno de ejecución.")

## Después, en la estación

1. Guarda el zip **fuera del repositorio**, por ejemplo en `D:\moldesign-dev-data\colab-plddt\`, y descomprímelo.
2. Comprueba qué diría hoy el producto de cada archivo. Es la línea base del cambio:

   ```
   python-embed\python.exe -I scripts\colab_plddt\leer_con_el_producto.py D:\moldesign-dev-data\colab-plddt\<carpeta>
   ```

3. Con `CONVENCION.md` y `analisis.json` se decide qué grupos quedan verificados. El cambio en
   `backend/services/targets/prediccion.py` llevará como fixtures recortes de estos archivos, con su procedencia y
   sus hashes, igual que se hizo con AlphaFold DB. Lo que no cumpla el criterio seguirá saliendo «no disponible».